# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cust40078-sudo/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Directory setup for exports
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../figures', exist_ok=True)

print("Setup completed. Export directories ready.")

Setup completed. Export directories ready.


## 1. Ranked Actions + Reason Codes

### Strategy & Archetype Mapping
We map validated model outputs to clear, actionable decisions. Each action is paired with a transparent reason code to build trust with human reviewers.

| Archetype / Trigger | Primary Action | Reason Code | Description |
| :--- | :--- | :--- | :--- |
| **High Score (>= 0.80)** | `PROMOTE_HERO` | `RC_HIGH_PERF` | Top model score; feature in primary content slots. |
| **Moderate Score + High Decay** | `REFRESH_CONTENT` | `RC_DECAY_WARN` | Engagement is decaying rapidly; update copy/assets. |
| **Moderate Score + Low Decay** | `MAINTAIN` | `RC_STABLE` | Consistent performance; retain current positioning. |
| **Low Score (< 0.40)** | `ARCHIVE` | `RC_LOW_ENG` | Low predicted engagement; schedule for deprecation. |
| **High Risk Flag** | `HUMAN_REVIEW` | `RC_RISK_FLAG` | Security/compliance flag detected; requires human audit. |

In [2]:
# Generate synthetic validated queue data
np.random.seed(42)
n_items = 100

data = {
    'content_id': [f'CNT_{i:04d}' for i in range(1, n_items + 1)],
    'predicted_score': np.round(np.random.uniform(0.15, 0.98, n_items), 4),
    'decay_rate': np.round(np.random.uniform(0.02, 0.45, n_items), 4),
    'risk_flag': np.random.choice([0, 1], size=n_items, p=[0.92, 0.08])
}

df = pd.DataFrame(data)

def assign_action(row):
    if row['risk_flag'] == 1:
        return 'HUMAN_REVIEW', 'RC_RISK_FLAG', 'Flagged for risk/compliance audit'
    elif row['predicted_score'] >= 0.80:
        return 'PROMOTE_HERO', 'RC_HIGH_PERF', 'High predicted engagement score'
    elif row['predicted_score'] >= 0.50 and row['decay_rate'] > 0.25:
        return 'REFRESH_CONTENT', 'RC_DECAY_WARN', 'Moderate score with high engagement decay'
    elif row['predicted_score'] >= 0.40:
        return 'MAINTAIN', 'RC_STABLE', 'Stable predicted engagement'
    else:
        return 'ARCHIVE', 'RC_LOW_ENG', 'Low predicted engagement score'

df[['action', 'reason_code', 'reason_desc']] = df.apply(assign_action, axis=1, result_type='expand')

# Rank by score descending
df_ranked = df.sort_values(by='predicted_score', ascending=False).reset_index(drop=True)

print("--- Top 5 Ranked Queue Items ---")
print(df_ranked[['content_id', 'predicted_score', 'action', 'reason_code']].head())

--- Top 5 Ranked Queue Items ---
  content_id  predicted_score        action   reason_code
0   CNT_0070           0.9691  PROMOTE_HERO  RC_HIGH_PERF
1   CNT_0012           0.9550  PROMOTE_HERO  RC_HIGH_PERF
2   CNT_0051           0.9548  PROMOTE_HERO  RC_HIGH_PERF
3   CNT_0035           0.9515  PROMOTE_HERO  RC_HIGH_PERF
4   CNT_0002           0.9391  PROMOTE_HERO  RC_HIGH_PERF


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended Use and Limits

### Scope & Boundaries
- **Intended Purpose**: Decision-support tool for editorial and content operations teams to prioritize batch content updates.
- **Valid Operational Window**: Recommendations are valid for a maximum of 14 days post-generation due to feature decay.
- **Out-of-Scope / Limits**:
  - Not designed for real-time automated publishing or live content deletion without oversight.
  - Not validated for unmonitored deployment across external third-party syndicate feeds.

In [4]:
# Documenting metadata limits
limits_metadata = {
    "intended_user": "Content Strategy & Operations Team",
    "decision_nature": "Decision-support / Semi-automated prioritization",
    "validity_window_days": 14,
    "max_batch_size": 1000,
    "disallowed_contexts": ["Automated Deletion", "Real-time Bidding", "Unmonitored Publishing"]
}

print("Intended Use & Limits Configured:")
print(json.dumps(limits_metadata, indent=2))

Intended Use & Limits Configured:
{
  "intended_user": "Content Strategy & Operations Team",
  "decision_nature": "Decision-support / Semi-automated prioritization",
  "validity_window_days": 14,
  "max_batch_size": 1000,
  "disallowed_contexts": [
    "Automated Deletion",
    "Real-time Bidding",
    "Unmonitored Publishing"
  ]
}


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human Review + The No-Go List

### Safeguards & Automation Boundaries
Certain edge cases require mandatory human intervention.

#### No-Go Rules (What Must NEVER Be Fully Automated):
1. **NO-GO 1**: Never automatically archive or delete assets with high historical traffic.
2. **NO-GO 2**: Never bypass manual review on items flagged with `RC_RISK_FLAG`.
3. **NO-GO 3**: Never execute auto-promotion on campaigns exceeding predefined budget limits.

In [6]:
# Extract Human Review Queue
human_review_queue = df_ranked[df_ranked['action'] == 'HUMAN_REVIEW'].copy()

print(f"Total items routed to Human Review: {len(human_review_queue)}")
print(human_review_queue[['content_id', 'predicted_score', 'reason_code', 'reason_desc']])

Total items routed to Human Review: 8
   content_id  predicted_score   reason_code  \
31   CNT_0046           0.6999  RC_RISK_FLAG   
41   CNT_0049           0.6038  RC_RISK_FLAG   
45   CNT_0048           0.5817  RC_RISK_FLAG   
47   CNT_0042           0.5610  RC_RISK_FLAG   
69   CNT_0062           0.3752  RC_RISK_FLAG   
72   CNT_0027           0.3157  RC_RISK_FLAG   
83   CNT_0041           0.2513  RC_RISK_FLAG   
88   CNT_0057           0.2234  RC_RISK_FLAG   

                          reason_desc  
31  Flagged for risk/compliance audit  
41  Flagged for risk/compliance audit  
45  Flagged for risk/compliance audit  
47  Flagged for risk/compliance audit  
69  Flagged for risk/compliance audit  
72  Flagged for risk/compliance audit  
83  Flagged for risk/compliance audit  
88  Flagged for risk/compliance audit  


In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / Retrain Triggers

### Model Health & Decay Metrics
To ensure recommendations do not go stale, we monitor feature drift and performance degradation against pre-set threshold triggers.

| Metric | Threshold Trigger | Action Required |
| :--- | :--- | :--- |
| **Data Drift (PSI)** | Population Stability Index > 0.25 | Trigger dataset re-sampling & retrain |
| **Performance Loss** | Prediction Error (MAE) increases by > 15% | Pause auto-queue & notify ML Ops |
| **Time Decay** | 14 Days elapsed since last model fit | Scheduled batch retraining run |

In [8]:
monitoring_triggers = {
    "psi_threshold": 0.25,
    "mae_degradation_pct": 15.0,
    "retrain_interval_days": 14,
    "alert_channel": "#ml-ops-alerts",
    "status": "ACTIVE"
}

with open('../outputs/monitoring_triggers.json', 'w') as f:
    json.dump(monitoring_triggers, f, indent=4)

print("Monitoring configuration exported to work/outputs/monitoring_triggers.json")

Monitoring configuration exported to work/outputs/monitoring_triggers.json


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the Paper

Exporting generated action queue, metrics summary, and figure artifacts to `work/outputs/` and `work/figures/`.

In [10]:
# 1. Export Ranked Action Queue CSV
df_ranked.to_csv('../outputs/ranked_action_queue.csv', index=False)

# 2. Save Metrics Summary JSON
metrics_summary = {
    "total_processed_items": len(df_ranked),
    "action_breakdown": df_ranked['action'].value_counts().to_dict(),
    "human_review_count": len(human_review_queue)
}

with open('../outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=4)

# 3. Generate & Save Action Distribution Figure
plt.figure(figsize=(8, 4.5))
sns.countplot(data=df_ranked, x='action', palette='viridis')
plt.title('Content Action Distribution Queue')
plt.xlabel('Assigned Action Archetype')
plt.ylabel('Item Count')
plt.tightlayout() if hasattr(plt, 'tightlayout') else plt.tight_layout()
plt.savefig('../figures/action_distribution.png', dpi=300)
plt.close()

print("Exports complete:")
print(" - CSV: work/outputs/ranked_action_queue.csv")
print(" - JSON: work/outputs/playbook_metrics.json")
print(" - JSON: work/outputs/monitoring_triggers.json")
print(" - PNG: work/figures/action_distribution.png")

/tmp/ipykernel_4714/594607536.py:16: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.countplot(data=df_ranked, x='action', palette='viridis')


Exports complete:
 - CSV: work/outputs/ranked_action_queue.csv
 - JSON: work/outputs/playbook_metrics.json
 - JSON: work/outputs/monitoring_triggers.json
 - PNG: work/figures/action_distribution.png


In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.